In [ ]:
#to prevent colab to automatically disconnect
import IPython
from google.colab import output

display(IPython.display.Javascript('''
 function ClickConnect(){
   btn = document.querySelector("colab-connect-button")
   if (btn != null){
     console.log("Click colab-connect-button");
     btn.click()
     }

   btn = document.getElementById('ok')
   if (btn != null){
     console.log("Click reconnect");
     btn.click()
     }
  }

setInterval(ClickConnect,60000)
'''))

print("Done.")

<IPython.core.display.Javascript object>

Done.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **Step 1:Importing Libraries and Installing Dependencies**

In [ ]:
pip install keras-self-attention

  Preparing metadata (setup.py) ... done
  Created wheel for keras-self-attention: filename=keras_self_attention-0.51.0-py3-none-any.whl size=18895 sha256=2021cc6dab0d0cd1e33551a57886e7cb4ddd350060f3e4b2fc1acdf00e060b5d
  Stored in directory: /root/.cache/pip/wheels/46/f9/96/709295c836133071c12a300729fed4027757f889c01695feea
Successfully built keras-self-attention


In [ ]:
pip install tensorflow

In [ ]:
# For data processing
import numpy as np
import math
from math import sqrt

# For data processing and manipulation
import pandas as pd
import csv

# For date calculations
import datetime
import time

# For ploting data
import IPython
import IPython.display

import itertools
from itertools import cycle
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# For checking path
import os , gc
import csv
import json


from scipy.stats import hmean

from sklearn.metrics import mean_squared_error, mean_absolute_error

#tensorflow libs
from tensorflow import keras
from tensorflow.keras.callbacks import EarlyStopping , Callback
import tensorflow as tf
from tensorflow.keras import backend as K
from keras import backend

from tensorflow.keras.layers import *
from tensorflow.keras.layers import Dense , GRU ,Dropout , PReLU , RepeatVector ,TimeDistributed, Attention,LayerNormalization,Add
from tensorflow.keras.models import Sequential,load_model
from tensorflow.keras.utils import to_categorical , plot_model
from tensorflow.keras import regularizers, constraints, initializers, activations
#from keras.layers.recurrent import Recurrent, _time_distributed_dense
from tensorflow.keras.layers import SimpleRNN as Recurrent
#from tensorflow.compat.v1.keras.layers import RNN

from tensorflow.keras.layers import InputSpec

from keras_self_attention import SeqSelfAttention
from tensorflow.keras.layers import Concatenate

from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, Bidirectional, TimeDistributed, LayerNormalization, Add
from tensorflow.keras import layers


tf.get_logger().setLevel('ERROR')
mpl.rcParams['figure.figsize'] = (8, 6)
mpl.rcParams['axes.grid'] = False

# **Step 2: Loading Dataset**

In [ ]:
#(paths in my personal drive)
model_save_path = r'/content/drive/MyDrive/Crime Prediction/LA/Project Tasks(Codes)/Forecasting(without weather dataset)/3.Crime Forecasting/Trained_model_Files'
dataset_path = r'/content/drive/MyDrive/Crime Prediction/LA/Project Tasks(Codes)/Forecasting(without weather dataset)/2.Preparing Dataset to feed Model/patrol_wise_crime_dataset'




In [ ]:
files = os.listdir(dataset_path)
files

['Central_1.csv',
 'Rampart_2.csv',
 'Southwest_3.csv',
 'Hollenbeck_4.csv',
 'Harbor_5.csv',
 'Hollywood_6.csv',
 'Wilshire_7.csv',
 'West LA_8.csv',
 'Van Nuys_9.csv',
 'West Valley_10.csv',
 'Northeast_11.csv',
 '77th Street_12.csv',
 'Newton_13.csv',
 'Pacific_14.csv',
 'N Hollywood_15.csv',
 'Foothill_16.csv',
 'Devonshire_17.csv',
 'Southeast_18.csv',
 'Mission_19.csv',
 'Olympic_20.csv',
 'Topanga_21.csv']

In [ ]:
#loading dataset
patrol_divisons = {}
dataset = {}
for file in files:
  name = file.split('_')[0]
  id = file.split('_')[1].split('.')[0]
  patrol_divisons[int(id)] = name
  dataset[int(id)] = pd.read_csv(os.path.join(dataset_path,file))


patrol_divisons = dict(sorted(patrol_divisons.items()))
patrol_divisons

{1: 'Central',
 2: 'Rampart',
 3: 'Southwest',
 4: 'Hollenbeck',
 5: 'Harbor',
 6: 'Hollywood',
 7: 'Wilshire',
 8: 'West LA',
 9: 'Van Nuys',
 10: 'West Valley',
 11: 'Northeast',
 12: '77th Street',
 13: 'Newton',
 14: 'Pacific',
 15: 'N Hollywood',
 16: 'Foothill',
 17: 'Devonshire',
 18: 'Southeast',
 19: 'Mission',
 20: 'Olympic',
 21: 'Topanga'}

In [ ]:
columns_in_dataset = dataset[1].columns
columns_in_dataset

Index(['Unnamed: 0', 'datetime', 'p_id', '1', '2', '3', '4', '5', '6', '7',
       '8', 'group 0', 'count', 'day sin', 'day cos', 'week sin', 'week cos',
       'year sin', 'year cos'],
      dtype='object')

In [ ]:
for key in dataset:
    if 'Unnamed: 0' in dataset[key].columns:
        dataset[key] = dataset[key].drop('Unnamed: 0', axis=1)


In [ ]:
dataset[1].head()

,datetime,p_id,1,2,3,4,5,6,7,8,group 0,count,day sin,day cos,week sin,week cos,year sin,year cos
0,2010-01-01 00:00:00,1,6.0,1.0,0.0,3.0,0.0,0.0,0.0,1.0,0.0,11.0,-4.416858e-12,1.000000e+00,0.781831,0.623490,0.005161,0.999987
1,2010-01-01 03:00:00,1,2.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,3.0,7.071068e-01,7.071068e-01,0.846724,0.532032,0.007311,0.999973
2,2010-01-01 06:00:00,1,0.0,0.0,0.0,12.0,1.0,0.0,0.0,1.0,0.0,14.0,1.000000e+00,6.980203e-12,0.900969,0.433884,0.009461,0.999955
3,2010-01-01 09:00:00,1,2.0,0.0,0.0,13.0,1.0,0.0,1.0,0.0,0.0,17.0,7.071068e-01,-7.071068e-01,0.943883,0.330279,0.011612,0.999933
4,2010-01-01 12:00:00,1,0.0,2.0,0.0,7.0,0.0,0.0,0.0,1.0,0.0,10.0,9.543547e-12,-1.000000e+00,0.974928,0.222521,0.013762,0.999905


In [ ]:
print(dataset[1]["count"].max())
print(dataset[1]["count"].min())

69.0
0.0


# **Step 3: Converting to Time Series data**

**Utility Functions**

In [ ]:
def train_test_val_split(dataset):
  columns_indices = {name:i for i,name in enumerate(dataset.columns)}
  n = len(dataset)

  #splitting dataset
  training_set = dataset[:int(n*0.7)]
  validation_set = dataset[int(n*0.7):int(n*0.9)]
  test_set = dataset[int(n*0.9):]

  num_features = dataset.shape[1]
  return training_set, validation_set, test_set, num_features, columns_indices

In [ ]:
class WindowGenerator():

    def __init__(self, input_width, label_width, shift,
               train_df, val_df, test_df,
               label_columns=None , shuffle=False , batch_size = 64):
        '''
        The __init__ method includes all the necessary logic for the input and label indices.
        Input:
            input_width : input width / window size
            label_width : output width
            shift : size of window shifting forward
            train_df : train dataset
            val_df : validation dataset
            test_df : test dataset
            label_columns ( Default = None) : Label Columns
            shuffle ( Default = False) : weather to shuffle data
            batch_size (Default = 64) : Batch Size
        Output: None
        Example :
            w2 = WindowGenerator(input_width=6, label_width=1, shift=1,
                     label_columns=['count'])
            w2

        '''
        # Store the raw data.
        self.train_df = train_df
        self.val_df = val_df
        self.test_df = test_df
        self.shuffle = shuffle
        self.batch_size = batch_size

        # Work out the label column indices.
        self.label_columns = label_columns
        if label_columns is not None:
            self.label_columns_indices = {name: i for i, name in
                                        enumerate(label_columns)}

        self.column_indices = {name: i for i, name in
                            enumerate(train_df.columns)}


        # Work out the window parameters.
        self.input_width = input_width
        self.label_width = label_width
        self.shift = shift

        self.total_window_size = input_width + shift

        self.input_slice = slice(0, input_width) #(start , stop)
        self.input_indices = np.arange(self.total_window_size)[self.input_slice]

        self.label_start = self.total_window_size - self.label_width
        self.labels_slice = slice(self.label_start, None)
        self.label_indices = np.arange(self.total_window_size)[self.labels_slice]

    def __repr__(self):
        return '\n'.join([
            f'Total window size: {self.total_window_size}',
            f'Input indices: {self.input_indices}',
            f'Label indices: {self.label_indices}',
            f'Label column name(s): {self.label_columns}'])

    def split_window(self, features):

        inputs = features[:, self.input_slice, :]
        labels = features[:, self.labels_slice, :]
        #taking only the labels that are presentin the label_columns
        if self.label_columns is not None:
            labels = tf.stack([labels[:, :, self.column_indices[name]] for name in self.label_columns],axis=-1)

        # Slicing doesn't preserve static shape information, so set the shapes
        # manually. This way the `tf.data.Datasets` are easier to inspect.
        inputs.set_shape([None, self.input_width, None])
        labels.set_shape([None, self.label_width, None])

        return inputs, labels



    def make_dataset(self, data):

        data = np.array(data, dtype=np.float32)
        ds = tf.keras.preprocessing.timeseries_dataset_from_array(
            data=data,
            targets=None,
            sequence_length=self.total_window_size,
            sequence_stride=1,
            shuffle=self.shuffle,
            batch_size=self.batch_size,)
        ds = ds.map(self.split_window)
        return ds


    def create_dataset2(self , map_df , reshape=True):
      x = []
      y = []
      for res in iter(map_df):
        inputs, labels = res
        if(len(inputs)==64):
          x.append(inputs)
          y.append(labels)

      x = np.array(x)
      y = np.array(y)
      if(reshape):
        x = x.reshape(-1, x.shape[-2] , x.shape[-1])
        y = y.reshape(-1 , y.shape[-2] , y.shape[-1])
      return x , y

    '''
    properties for accessing  training, validation and test data as tf.data.Datasets using the above make_dataset method.
    Also a standard example batch for easy access and plotting
    '''

    @property
    def train(self):
        return self.make_dataset(self.train_df)

    @property
    def val(self):
        return self.make_dataset(self.val_df)

    @property
    def test(self):
        return self.make_dataset(self.test_df)



In [ ]:
def create_data(train , test , val , columns):
    '''
    Create dataset from main train , test , val with given columns
    '''
    if(columns==None):
        columns = train.columns
    new_train = train[columns]
    new_test = test[columns]
    new_val = val[columns]
    return new_train , new_test , new_val

In [ ]:
def save_history(history , path):
    # convert the history.history dict to a pandas DataFrame:
    hist_df = pd.DataFrame(history.history)
    # or save to csv:
    hist_csv_file = path
    with open(hist_csv_file, mode='w') as f:
        hist_df.to_json(f)

def get_history(path):
	with open(path) as json_file:
		data = json.load(json_file)
		return data

def save_model_weights(model , path):
  model.save_weights(path)


Functions for Model Compiling and Fitting

In [ ]:
def compileModel(model):

        lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
          initial_learning_rate=1e-4,
          decay_steps=1000,
          decay_rate=0.96,
          staircase=True,
      )
        optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)

        model.compile(loss=tf.losses.MeanAbsoluteError(),optimizer = optimizer,
                   metrics =[ tf.keras.metrics.RootMeanSquaredError(name='rmse'),
                              tf.keras.metrics.MeanAbsoluteError(name='mae')])
        return model

MAX_EPOCHS = 30

def fit(model,ROOTPATH, modelPath , historyPath , window=None, name=None , patience = 5):
    '''
    Compile and fit a model
    '''

    modelPathPar = os.path.join(ROOTPATH,modelPath)
    historyPathPar = os.path.join(ROOTPATH,historyPath)

    #Creating a file to store model and it's history
    if (name!=None):
        model_path = os.path.join(modelPathPar,name+".keras")
        history_path = os.path.join(historyPathPar,name+".json")

        if not os.path.exists(modelPathPar):
            os.makedirs(modelPathPar)
        if not os.path.exists(historyPathPar):
            os.makedirs(historyPathPar)

    if (name!=None and os.path.exists(model_path)):
      print("Loaded Pre Trained Model")
      modelOld = tf.keras.models.load_model(model_path)
      model.set_weights(modelOld.get_weights())
      del modelOld
      history = get_history(history_path)
      return model , history



    early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=patience)

    history = model.fit(window.train, epochs=MAX_EPOCHS,batch_size = 64,validation_data=window.val,
                        callbacks=[early_stopping],verbose=1)


    if(name!=None):
      model.save(model_path)
      save_history(history , history_path)

    return model , history.history

Function to save  metric values

In [ ]:
def build_metrics_dataframe(test_metrics_dict, val_metrics_dict, patrol_divisions_dict):
    """
    Converts performance dictionaries into a structured DataFrame.

    Parameters:
        test_metrics_dict (dict): Dict of test evaluation results {division_id: [loss, rmse, mae]}
        val_metrics_dict (dict): Dict of val evaluation results {division_id: [loss, rmse, mae]}
        patrol_divisions_dict (dict): Dict mapping division_id to division name

    Returns:
        pd.DataFrame: DataFrame with division names as rows and metrics as columns
    """
    rows = []

    for div_id in test_metrics_dict.keys():
        div_name = patrol_divisions_dict[int(div_id)]
        test = test_metrics_dict[div_id]
        val = val_metrics_dict[div_id]

        rows.append({
            "Division": div_name,
            "test rmse": test[1],
            "test mae": test[2],
            "val rmse": val[1],
            "val mae": val[2]
        })

    df = pd.DataFrame(rows)
    df.set_index("Division", inplace=True)
    return df


In [ ]:
#without temperature
general_indexs = ['1', '2', '3', '4', '5', '6', '7','8', 'count',
           'day sin', 'day cos', 'week sin', 'week cos',
           'year sin', 'year cos', 'group 0']

x_col = ['day sin' , 'day cos' , 'year sin' , 'year cos' , 'week cos' , 'week sin' ,'datetime']

def generate_window(df_now, ret_test = 0):
    train_df , val_df , test_df , num_features_df , column_indices_df = train_test_val_split(df_now)
    train_df , test_df , val_df = create_data(train_df , test_df , val_df , general_indexs)
    y_col = []

    #storing label columns
    for i in train_df.columns:
        if (i in x_col):
            continue
        y_col.append(i)

    #creating window generator object
    wide_window_all = WindowGenerator(train_df=train_df, test_df=test_df , val_df=val_df,
        input_width=24, label_width=24, shift=1,
        label_columns=y_col)

    if (ret_test == 1):
      return wide_window_all, test_df
    else:
      return wide_window_all


**1.Multi Head Attention Bi GRU**

In [ ]:
# Input layer
inputs = keras.Input(shape=(24, 16), name="gru_input")

# First BiGRU layer
x = layers.Bidirectional(
    layers.GRU(128, return_sequences=True, activation='swish'),
    name="bigru1"
)(inputs)
x = layers.Dropout(0.3, name="dropout_bigru")(x)

# Multi-Head Attention Layer (Self-Attention)
attn_output = layers.MultiHeadAttention(
    num_heads=4, key_dim=32, name="multihead_attention_gru"
)(x, x)
attn_output = layers.Dropout(0.3, name="dropout_attention_gru")(attn_output)
attn_output = layers.LayerNormalization(epsilon=1e-6, name="attn_residual_norm_gru")(x + attn_output)  # Residual connection

# Feed-Forward Layer after attention
ff = layers.TimeDistributed(
    layers.Dense(64, activation='swish'), name="ff1_gru"
)(attn_output)
ff = layers.Dropout(0.3, name="dropout_ff_gru")(ff)

# Project attention output to match ff shape for residual connection
attn_proj = layers.TimeDistributed(
    layers.Dense(64), name="attn_proj_gru"
)(attn_output)

# Final residual + normalization
ff_output = layers.LayerNormalization(epsilon=1e-6, name="ff_residual_norm_gru")(attn_proj + ff)

# Final Dense output layer (per time step)
output = layers.TimeDistributed(
    layers.Dense(10), name="output_layer_gru"
)(ff_output)

# Define the model
mh_attn_bi_gru = keras.Model(inputs, output, name="BiGRU_MHAttention_Residual")
mh_attn_bi_gru.summary()

Model: "BiGRU_MHAttention_Residual"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gru_input (InputLayer)    │ (None, 24, 16)         │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ bigru1 (Bidirectional)    │ (None, 24, 256)        │        112,128 │ gru_input[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_bigru (Dropout)   │ (None, 24, 256)        │              0 │ bigru1[0][0]           │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ multihead_attention_gru   │ (None, 24, 256)        │        131,712 │ dropout_bigru[0][0],   │
│ (MultiHeadAttention)      │                        │                │ dropout_bigru[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_attention_gru     │ (None, 24, 256)        │              0 │ multihead_attention_g… │
│ (Dropout)                 │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add (Add)                 │ (None, 24, 256)        │              0 │ dropout_bigru[0][0],   │
│                           │                        │                │ dropout_attention_gru… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ attn_residual_norm_gru    │ (None, 24, 256)        │            512 │ add[0][0]              │
│ (LayerNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ ff1_gru (TimeDistributed) │ (None, 24, 64)         │         16,448 │ attn_residual_norm_gr… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ attn_proj_gru             │ (None, 24, 64)         │         16,448 │ attn_residual_norm_gr… │
│ (TimeDistributed)         │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_ff_gru (Dropout)  │ (None, 24, 64)         │              0 │ ff1_gru[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add_1 (Add)               │ (None, 24, 64)         │              0 │ attn_proj_gru[0][0],   │
│                           │                        │                │ dropout_ff_gru[0][0]   │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ ff_residual_norm_gru      │ (None, 24, 64)         │            128 │ add_1[0][0]            │
│ (LayerNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ output_layer_gru          │ (None, 24, 10)         │            650 │ ff_residual_norm_gru[… │
│ (TimeDistributed)         │                        │                │                        │
└───────────────────────────┴────────────────────────┴────────────────┴────────────────────────┘

 Total params: 278,026 (1.06 MB)

 Trainable params: 278,026 (1.06 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
val_performance_of_mh_attn_bi_gru = {}
performance_of_mh_attn_bi_gru = {}
histories_of_mh_attn_bi_gru = {}
mh_attn_bi_gru_model = compileModel(mh_attn_bi_gru)
save_path = os.path.join(model_save_path,"Multi Head Attention Bi GRU")
for x in patrol_divisons.keys():
        wide_window_all = generate_window(dataset[x])
        print(x,patrol_divisons[x])
        x = str(x)
        trained_mh_attn_bi_gru_model , history  = fit(
            mh_attn_bi_gru_model,save_path, "mh_attn_bi_gru_model_files" , "mh_attn_bi_gru_history_files" , name='mh_attn_bi_gru_all_feature_'+x , window=wide_window_all)
        histories_of_mh_attn_bi_gru['bd_'+x] = history
        val_performance_of_mh_attn_bi_gru[x] = trained_mh_attn_bi_gru_model.evaluate(wide_window_all.val)
        performance_of_mh_attn_bi_gru[x] = trained_mh_attn_bi_gru_model.evaluate(wide_window_all.test)


1 Central
Epoch 1/30
483/483 ━━━━━━━━━━━━━━━━━━━━ 44s 50ms/step - loss: 0.7299 - mae: 0.7299 - rmse: 1.1179 - val_loss: 0.7155 - val_mae: 0.7155 - val_rmse: 1.3701
Epoch 2/30
483/483 ━━━━━━━━━━━━━━━━━━━━ 13s 15ms/step - loss: 0.4727 - mae: 0.4727 - rmse: 0.8292 - val_loss: 0.6917 - val_mae: 0.6917 - val_rmse: 1.1175
Epoch 3/30
483/483 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - loss: 0.3958 - mae: 0.3958 - rmse: 0.6699 - val_loss: 0.4910 - val_mae: 0.4910 - val_rmse: 0.9096
Epoch 4/30
483/483 ━━━━━━━━━━━━━━━━━━━━ 7s 15ms/step - loss: 0.3238 - mae: 0.3238 - rmse: 0.5969 - val_loss: 0.4443 - val_mae: 0.4443 - val_rmse: 0.8140
Epoch 5/30
483/483 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step - loss: 0.2916 - mae: 0.2916 - rmse: 0.5574 - val_loss: 0.4234 - val_mae: 0.4234 - val_rmse: 0.7771
Epoch 6/30
483/483 ━━━━━━━━━━━━━━━━━━━━ 10s 14ms/step - loss: 0.2602 - mae: 0.2602 - rmse: 0.5071 - val_loss: 0.4261 - val_mae: 0.4261 - val_rmse: 0.7546
Epoch 7/30
483/483 ━━━━━━━━━━━━━━━━━━━━ 10s 14ms/step - loss: 0.2563

In [ ]:
# Call the function after training is done
mh_attn_bi_gru_results_df = build_metrics_dataframe(
     performance_of_mh_attn_bi_gru,
    val_performance_of_mh_attn_bi_gru,
    patrol_divisons
)
csv_path = os.path.join(save_path,"mh_attn_bi_gru_results_1.csv")
mh_attn_bi_gru_results_df.to_csv(csv_path)


In [ ]:
mh_attn_bi_gru_results_df.head(21)

,test rmse,test mae,val rmse,val mae
Division,,,,
Central,0.713613,0.223138,0.476545,0.184686
Rampart,0.403934,0.101825,0.308708,0.094344
Southwest,0.363826,0.115266,0.484089,0.121378
Hollenbeck,0.184826,0.061920,0.237104,0.070740
Harbor,0.194903,0.054310,0.246436,0.061823
Hollywood,0.250244,0.065559,0.304336,0.083097
Wilshire,0.227689,0.051474,0.265104,0.065106
West LA,0.202879,0.045108,0.256146,0.062406
Van Nuys,0.197093,0.045592,0.232320,0.055161


**Testing model trained on Central patrol divison**

In [ ]:
model = load_model(r'/content/drive/MyDrive/Crime Prediction/LA/Project Tasks(Codes)/Forecasting/3.Multi-Modal Fusion/FLF/Trained_model_files/Multi Head Attention Bi GRU/mh_attn_bi_gru_model_files/mh_attn_bi_gru_all_feature_3.keras')

In [ ]:
wide_window = generate_window(dataset[3])

In [ ]:
test_data = wide_window.test

In [ ]:

# Unbatch and convert to NumPy arrays
x_test_list = []
y_test_list = []

for x, y in test_data.unbatch():
    x_test_list.append(x.numpy())
    y_test_list.append(y.numpy())

# Convert lists to NumPy arrays
x_test = np.array(x_test_list)
y_test = np.array(y_test_list)

print("x_test shape:", x_test.shape)
print("y_test shape:", y_test.shape)

x_test shape: (4178, 24, 16)
y_test shape: (4178, 24, 10)


In [ ]:
# Get predicted y values
y_pred = model.predict(x_test)
y_pred = np.round(np.maximum(y_pred, 0))
print("y_pred shape:", y_pred.shape)


131/131 ━━━━━━━━━━━━━━━━━━━━ 8s 26ms/step
y_pred shape: (4178, 24, 10)


In [ ]:
# Number of samples you want to inspect
num_samples = 5

# Create a list to store the comparison DataFrames
comparison_list = []

# Loop over the first `num_samples`
for i in range(num_samples):
    true_vals = y_test[i]   # shape: (24, 10)
    pred_vals = y_pred[i]   # shape: (24, 10)

    # Column names for better context
    columns = ['1','2','3','4','5','6','7','8','Count','Group 0']  # Adjust if needed

    df_true = pd.DataFrame(true_vals, columns=[f"{col}" for col in columns])
    df_pred = pd.DataFrame(pred_vals, columns=[f"{col}" for col in columns])

    df_both = pd.concat([df_true, df_pred], axis=1)
    comparison_list.append(df_both)

# Combine all comparisons into one DataFrame
full_comparison = pd.concat(comparison_list, axis=0)

# Reset index
full_comparison.reset_index(drop=True, inplace=True)



In [ ]:
full_comparison.head()

,1,2,3,4,5,6,7,8,Count,Group 0,1,2,3,4,5,6,7,8,Count,Group 0
0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,3.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,3.0,0.0
1,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,3.0,0.0
2,2.0,1.0,0.0,2.0,1.0,0.0,1.0,0.0,7.0,0.0,2.0,1.0,0.0,2.0,1.0,0.0,0.0,0.0,7.0,0.0
3,0.0,0.0,0.0,6.0,0.0,1.0,0.0,0.0,7.0,0.0,0.0,0.0,0.0,6.0,0.0,1.0,0.0,0.0,7.0,0.0
4,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,2.0,0.0
